# 📚 Classification de Fleurs avec CNN

## 🎯 Ce Que Vous Allez Apprendre

- Construire un CNN pour la classification d'images multi-classes
- Charger et prétraiter des données avec `image_dataset_from_directory`
- Techniques de visualisation d'images
- Conception, compilation et entraînement de l'architecture du modèle
- Évaluer les performances du modèle avec des graphiques de précision et de perte

## 🛠️ Ce Que Vous Allez Créer

Un modèle CNN capable de classifier des images de 14 espèces de fleurs différentes avec une grande précision.

In [ ]:
# Partie 1: Exploration et Chargement des Données
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import numpy as np

# Charger le dataset de fleurs
import pathlib
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
data_dir = tf.keras.utils.get_file('flower_photos', origin=dataset_url, untar=True)
data_dir = pathlib.Path(data_dir)

# Compter le nombre total d'images
image_count = len(list(data_dir.glob('*/*.jpg')))
print(f"Nombre total d'images: {image_count}")

# Paramètres
img_height = 48  # Résolution réduite pour accélérer l'entraînement
img_width = 48
batch_size = 32

# Créer les ensembles d'entraînement et de validation (80/20 split)
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

# Obtenir les noms des classes
class_names = train_ds.class_names
print(f"Classes de fleurs: {class_names}")
print(f"Nombre de classes: {len(class_names)}")

# Visualiser quelques images
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
  for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[labels[i]])
    plt.axis("off")
plt.show()

# Partie 2: Architecture du Modèle CNN
# Normalisation des pixels
normalization_layer = layers.Rescaling(1./255)

# Créer le modèle CNN
model = keras.Sequential([
  # Normalisation
  normalization_layer,

  # Premier bloc convolutionnel
  layers.Conv2D(32, 3, activation='relu', padding='same'),
  layers.MaxPooling2D(),

  # Deuxième bloc convolutionnel
  layers.Conv2D(64, 3, activation='relu', padding='same'),
  layers.MaxPooling2D(),

  # Troisième bloc convolutionnel
  layers.Conv2D(128, 3, activation='relu', padding='same'),
  layers.MaxPooling2D(),

  # Dropout pour réduire le surapprentissage
  layers.Dropout(0.2),

  # Aplatir les features maps
  layers.Flatten(),

  # Couches denses
  layers.Dense(128, activation='relu'),
  layers.Dropout(0.2),
  layers.Dense(len(class_names), activation='softmax')  # Sortie pour 14 classes
])

# Afficher l'architecture
model.summary()

# Partie 3: Réglage des Hyperparamètres
# Compiler le modèle avec Adam optimizer
model.compile(
  optimizer='adam',
  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
  metrics=['accuracy']
)

# Entraîner le modèle
epochs = 10  # Nombre d'époques réduit pour l'exemple
history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

# Tester avec d'autres optimiseurs (optionnel)
print("\n--- Test avec RMSprop ---")
model_rmsprop = keras.models.clone_model(model)
model_rmsprop.compile(
  optimizer='rmsprop',
  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
  metrics=['accuracy']
)

print("\n--- Test avec SGD ---")
model_sgd = keras.models.clone_model(model)
model_sgd.compile(
  optimizer='sgd',
  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
  metrics=['accuracy']
)

# Partie 4: Augmentation des Données
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Créer un générateur d'augmentation de données
data_augmentation = keras.Sequential([
  layers.RandomFlip("horizontal_and_vertical"),
  layers.RandomRotation(0.2),
  layers.RandomZoom(0.2),
])

# Modèle avec augmentation
model_augmented = keras.Sequential([
  data_augmentation,  # Couche d'augmentation
  normalization_layer,
  layers.Conv2D(32, 3, activation='relu', padding='same'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, activation='relu', padding='same'),
  layers.MaxPooling2D(),
  layers.Conv2D(128, 3, activation='relu', padding='same'),
  layers.MaxPooling2D(),
  layers.Dropout(0.2),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  layers.Dropout(0.2),
  layers.Dense(len(class_names), activation='softmax')
])

model_augmented.compile(
  optimizer='adam',
  loss=tf.keras.losses.SparseCategoricalCrossentropy(),
  metrics=['accuracy']
)

# Entraîner avec augmentation
history_augmented = model_augmented.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

# Partie 5: Évaluation des Performances
# Visualiser la précision
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Époque')
plt.ylabel('Précision')
plt.title('Précision du Modèle')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Époque')
plt.ylabel('Perte')
plt.title('Perte du Modèle')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Comparer modèle de base vs avec augmentation
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['val_accuracy'], label='Sans Augmentation')
plt.plot(history_augmented.history['val_accuracy'], label='Avec Augmentation')
plt.xlabel('Époque')
plt.ylabel('Précision de Validation')
plt.title('Comparaison: Impact de l\'Augmentation')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['val_loss'], label='Sans Augmentation')
plt.plot(history_augmented.history['val_loss'], label='Avec Augmentation')
plt.xlabel('Époque')
plt.ylabel('Perte de Validation')
plt.title('Comparaison: Impact de l\'Augmentation')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Partie 6: Sauvegarde du Modèle
# Sauvegarder le meilleur modèle
model_augmented.save('flower_classification_model.h5')
print("Modèle sauvegardé avec succès!")

# Sauvegarder au format SavedModel (recommandé pour TensorFlow)
model_augmented.save('flower_classification_model')
print("Modèle sauvegardé au format SavedModel!")

# Afficher les résultats finaux
final_train_acc = history_augmented.history['accuracy'][-1]
final_val_acc = history_augmented.history['val_accuracy'][-1]
print(f"\nPrécision finale sur l'entraînement: {final_train_acc:.4f}")
print(f"Précision finale sur la validation: {final_val_acc:.4f}")